In [3]:
import os
from databricks import sql
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

def get_connection():
    return sql.connect(
        server_hostname = os.getenv("DATABRICKS_HOST"),
        http_path = os.getenv("DATABRICKS_HTTP"),
        access_token = os.getenv("DATABRICKS_TOKEN")
    )


def run_query(query: str):
    with get_connection() as conn:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()
            return pd.DataFrame(rows, columns=columns)

In [4]:
run_query("""
    SELECT trip_id, stop_name, stop_sequence, arrival_time, departure_time
    FROM transport.gold.gtfs_gold
    WHERE date = '2026-05-19'
    AND trip_id = (
        SELECT trip_id FROM transport.gold.gtfs_gold 
        WHERE date = '2026-05-19' 
        LIMIT 1
    )
    ORDER BY stop_sequence
""")

,trip_id,stop_name,stop_sequence,arrival_time,departure_time
0,OCESN12374F1187_F:OUI:FR:Line::BAF1DE9A-72EC-4...,Marseille Saint-Charles,0,16:04:00,16:04:00
1,OCESN12374F1187_F:OUI:FR:Line::BAF1DE9A-72EC-4...,Aix-en-Provence TGV,1,16:15:00,16:18:00
2,OCESN12374F1187_F:OUI:FR:Line::BAF1DE9A-72EC-4...,Avignon TGV,2,16:38:00,16:41:00
3,OCESN12374F1187_F:OUI:FR:Line::BAF1DE9A-72EC-4...,Paris Gare de Lyon Hall 1 - 2,3,19:22:00,19:22:00


In [3]:
df = run_query("SELECT * FROM transport.gold.mart_retards_tgv LIMIT 10")
print(df)

          gare_depart          gare_arrivee                      mois  \
0      MULHOUSE VILLE            PARIS LYON 2026-01-01 00:00:00+00:00   
1  PARIS MONTPARNASSE                 TOURS 2026-01-01 00:00:00+00:00   
2              ANNECY            PARIS LYON 2026-01-01 00:00:00+00:00   
3          PARIS LYON           DIJON VILLE 2026-01-01 00:00:00+00:00   
4               DOUAI            PARIS NORD 2026-01-01 00:00:00+00:00   
5         MACON LOCHE            PARIS LYON 2026-01-01 00:00:00+00:00   
6          PARIS NORD                 LILLE 2026-01-01 00:00:00+00:00   
7          PARIS LYON                GENEVE 2026-01-01 00:00:00+00:00   
8      LYON PART DIEU  MARSEILLE ST CHARLES 2026-01-01 00:00:00+00:00   
9      LYON PART DIEU            PARIS LYON 2026-01-01 00:00:00+00:00   

   retard_moyen_arrivee  retard_moyen_depart  total_annules  \
0              9.063538             3.485333              1   
1              5.771174             2.487212             11   
2      

In [4]:
df.head()

,gare_depart,gare_arrivee,mois,retard_moyen_arrivee,retard_moyen_depart,total_annules,prct_cause_externe,prct_cause_infrastructure,prct_cause_materiel
0,MULHOUSE VILLE,PARIS LYON,2026-01-01 00:00:00+00:00,9.063538,3.485333,1,45.454545,10.909091,12.727273
1,PARIS MONTPARNASSE,TOURS,2026-01-01 00:00:00+00:00,5.771174,2.487212,11,25.000000,20.833333,29.166667
2,ANNECY,PARIS LYON,2026-01-01 00:00:00+00:00,12.571046,1.525547,1,32.432432,16.216216,8.108108
3,PARIS LYON,DIJON VILLE,2026-01-01 00:00:00+00:00,6.245314,4.541402,2,31.578947,7.017544,29.824561
4,DOUAI,PARIS NORD,2026-01-01 00:00:00+00:00,10.021868,3.430149,14,27.777778,7.407407,12.962963


In [ ]:
import requests

geojson = requests.get("https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions-version-simplifiee.geojson").json()
noms_geojson = [f["properties"]["nom"] for f in geojson["features"]]

print("=== GeoJSON ===")
print(sorted(noms_geojson))

print("\n=== Tes données ===")
print(sorted(df["region"].unique().tolist()))

=== GeoJSON ===
['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne', 'Centre-Val de Loire', 'Corse', 'Grand Est', 'Hauts-de-France', 'Normandie', 'Nouvelle-Aquitaine', 'Occitanie', 'Pays de la Loire', "Provence-Alpes-Côte d'Azur", 'Île-de-France']

=== Tes données ===
['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne', 'Centre Val-de-Loire', 'Etoile Amiens', 'Grand Est', 'Hauts-de-France', 'Loire Océan', 'Normandie', 'Nouvelle Aquitaine', 'Occitanie', 'Pays-de-la-Loire', "Provence Alpes Côte d'Azur", 'Sud Azur']
